codigo de como hacer mas facil un degrade 


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.figsize'] = [10, 5]
size_imagen = 100

gradiente = np.linspace(0, 1, size_imagen)
gradiente = np.tile(gradiente, (size_imagen, 1)).T

imagen = plt.cm.viridis(gradiente)[:, :, :3] 
imagen = (imagen * 255).astype(np.uint8)

plt.imshow(imagen)
plt.axis("off")

cv2.imwrite('degradado.png', cv2.cvtColor(imagen, cv2.COLOR_RGB2BGR))
plt.show()

IMAGEN MPRO

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# CREAR IMAGEN
tam = 800
img = np.zeros((tam, tam, 3), dtype=np.uint8)
# CIELO (SIN FOR)
filas = np.linspace(0, 1, tam).reshape(tam, 1)

R = (255 * filas).astype(np.uint8)
G = (120 * filas).astype(np.uint8)
B = (255 * (1 - filas * 0.7)).astype(np.uint8)

img[:] = np.dstack((B, G, R))

# SOL 
Y, X = np.ogrid[:tam, :tam]

cx, cy = 400, 300
radio2 = 5000

mask_sol = (X - cx)**2 + (Y - cy)**2 < radio2
img[mask_sol] = (0, 220, 255)

# CERRO IZQUIERDO 
zona = (Y >= 400) & (Y < 650)

mask1 = zona & (X < Y - 300)
mask2 = zona & (X < Y - 350)

img[mask1] = (60, 60, 90)
img[mask2] = (40, 40, 70)

# CERRO DERECHO
zona2 = (Y >= 450) & (Y < 700)

mask3 = zona2 & (X > tam - (Y - 300))
img[mask3] = (30, 30, 60)

# REFLEJO 
mitad = 500
altura_reflejo = tam - mitad  
reflejo = img[mitad - altura_reflejo:mitad][::-1] * 0.7
img[mitad:] = reflejo.astype(np.uint8)

# FILTRO BAYER SIN FOR
def crear_bayer_fast(imagen, patron):
    B, G, R = cv2.split(imagen)
    mosaico = np.zeros(imagen.shape[:2], dtype=np.uint8)

    if patron == "RGGB":
        mosaico[0::2, 0::2] = R[0::2, 0::2]
        mosaico[0::2, 1::2] = G[0::2, 1::2]
        mosaico[1::2, 0::2] = G[1::2, 0::2]
        mosaico[1::2, 1::2] = B[1::2, 1::2]

    elif patron == "BGGR":
        mosaico[0::2, 0::2] = B[0::2, 0::2]
        mosaico[0::2, 1::2] = G[0::2, 1::2]
        mosaico[1::2, 0::2] = G[1::2, 0::2]
        mosaico[1::2, 1::2] = R[1::2, 1::2]

    elif patron == "GRBG":
        mosaico[0::2, 0::2] = G[0::2, 0::2]
        mosaico[0::2, 1::2] = R[0::2, 1::2]
        mosaico[1::2, 0::2] = B[1::2, 0::2]
        mosaico[1::2, 1::2] = G[1::2, 1::2]

    elif patron == "GBRG":
        mosaico[0::2, 0::2] = G[0::2, 0::2]
        mosaico[0::2, 1::2] = B[0::2, 1::2]
        mosaico[1::2, 0::2] = R[1::2, 0::2]
        mosaico[1::2, 1::2] = G[1::2, 1::2]

    return mosaico

def reconstruir(mosaico, patron):
    codigos = {
        "RGGB": cv2.COLOR_BAYER_RG2BGR,
        "BGGR": cv2.COLOR_BAYER_BG2BGR,
        "GRBG": cv2.COLOR_BAYER_GR2BGR,
        "GBRG": cv2.COLOR_BAYER_GB2BGR
    }
    return cv2.cvtColor(mosaico, codigos[patron])

# MOSTRAMS
patrones = ["RGGB", "BGGR", "GRBG", "GBRG"]

plt.figure(figsize=(12,8))

plt.subplot(2,3,1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Imagen optimizada")
plt.axis("off")

for i, p in enumerate(patrones):
    mosaico = crear_bayer_fast(img, p)
    rec = reconstruir(mosaico, p)

    plt.subplot(2,3,i+2)
    plt.imshow(cv2.cvtColor(rec, cv2.COLOR_BGR2RGB))
    plt.title(p)
    plt.axis("off")

    cv2.imwrite(f"bayer_{p}.png", rec)

plt.tight_layout()
plt.show()

cv2.imwrite("original.png", img)
